In [1]:
import html
import ipywidgets
import importlib
from pathlib import Path
import re
import sys

### Analyst

A notebook interface for asking questions about certain security conference proceedings using a prepared knowledge base. SQL queries can alternatively be used to search for researchers, talks, or tools. Answers to free-form questions are given from the pre-processed topic summaries. There are three dozen summaries covering topics across the conference sources. There are also summaries for each conference and for authors with more than one presentation in the data. The current dataset includes the PROMPT|GTFO Youtube channel and these recent conferences:


2025 conferences: CAMLIS, DEFCON, Blackhat BSides SF ; 2026 conferences: Unprompted.

- **Question** Answers questions using the pre-processed topic summaries from the conference list above. Summaries are also available for conferences and authors with multiple talks.
- **Query**  Runs deterministic annotation/database queries and does not call an LLM.
- **Status** checks the knowledge/index state in the SQLite data layer.


In [2]:

def find_analyst_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "the_analyst.py").exists():
            return path
    fallback = Path(r"C:\Users\flynn\Desktop\the-oracle")
    if (fallback / "the_analyst.py").exists():
        return fallback
    raise RuntimeError("Could not find the Analyst repo root. Start Jupyter from the repo or set the path manually.")

ROOT = find_analyst_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import the_analyst as analyst
from knowledge_indexing import knowledge_index as ki
from knowledge_agenting import topic_summarizer as ts
ts = importlib.reload(ts)

print(f"Analyst root: {ROOT}")

Analyst root: \\vmware-host\Shared Folders\SHARED\GitHub\analyst


In [3]:
# Core functions

def capture_analyst_print(fn, *args, **kwargs) -> str:
    """Capture Analyst output that normally goes through ki.safe_print."""
    lines: list[str] = []
    original_safe_print = ki.safe_print

    def notebook_print(value: str = "") -> None:
        lines.append(str(value))

    ki.safe_print = notebook_print
    try:
        fn(*args, **kwargs)
    finally:
        ki.safe_print = original_safe_print
    return "\n".join(lines).strip()


def ask_summaries_only(question: str, max_summaries: int = 3) -> str:
    """Ask using only routed summary evidence, with up to max_summaries loaded."""
    question = question.strip()
    if not question:
        return "Enter a question."
    max_summaries = max(1, int(max_summaries))
    evidence = analyst.tool_answer_from_summaries(question=question, max_topics=max_summaries)
    if evidence.startswith("No summary files found"):
        return evidence
    client = analyst.llm_client.create_client()
    response = client.messages.create(
        model=analyst.MODEL,
        max_tokens=analyst.QA_MAX_OUTPUT_TOKENS,
        system=analyst.AGENT_SYSTEM,
        messages=[
            {
                "role": "user",
                "content": (
                    f"Question: {question}\n\n"
                    "Answer using ONLY the summary evidence below. "
                    "Start with a Sources used line naming every summary file you relied on. "
                    "Then answer directly in natural prose. "
                    "If the summary evidence is thin or does not answer the question, say so plainly.\n\n"
                    f"{evidence}"
                ),
            }
        ],
    )
    return "\n".join(block.text for block in response.content if hasattr(block, "text")).strip()


def query_annotations(query: str, limit: int | None = 10) -> str:
    """Search structured annotations without using the LLM."""
    query = query.strip()
    if not query:
        return "Enter a query."
    return analyst.tool_query_annotations(query=query, limit=limit)


def analyst_status() -> str:
    return analyst.tool_get_status()


def list_summary_topics() -> str:
    """List topics that have classified records and can be summarized."""
    topics = ts.list_topics_for_summary(ki.DB_PATH)
    if not topics:
        return "No topics with classified records found."
    return "\n".join([*topics, f"topics: {len(topics)}"])


def list_topic_summaries() -> str:
    """List existing topic summary Markdown files."""
    if hasattr(ts, "list_existing_summaries"):
        summaries = ts.list_existing_summaries(ROOT / "summaries", "topic")
        if not summaries:
            return "No topic summary files found."
        lines = ["topic\tstatus\trecords\tgenerated_at\tpath"]
        for item in summaries:
            records = "" if item.records is None else str(item.records)
            lines.append(f"{item.name}\t{item.status}\t{records}\t{item.generated_at}\t{item.summary_path}")
        lines.append(f"summaries: {len(summaries)}")
        return "\n".join(lines)

    paths = sorted((ROOT / "summaries" / "topics").glob("*.md"))
    if not paths:
        return "No topic summary files found."
    lines = ["topic\tstatus\trecords\tgenerated_at\tpath"]
    for path in paths:
        lines.append(f"{path.stem}\tfile_only\t\t\t{path}")
    lines.append(f"summaries: {len(paths)}")
    return "\n".join(lines)

# Smoke check the no-LLM status path.
print(analyst_status())

Records by source:
  blackhat: 110
  bsideslv: 200
  bsidessf: 74
  camlis: 153
  defcon33: 320
  promptorgtfo: 67
  unprompted2026: 64
  total: 988

unclassified: 0
record annotations: 988/988
last import: 2026-06-23T14:10:17+00:00
last classified: 2026-06-23T15:47:28+00:00

Recent import log:
  2026-07-15T13:17:55+00:00 [INFO] candidate topics added: 4 (automation incident response, cloud detection pipeline, analysis bio signal, analysis specialized safety)
  2026-07-15T13:17:58+00:00 [INFO] candidate topics added: 1 (autonomous vulnerability discovery)
  2026-07-15T13:26:09+00:00 [INFO] candidate topics added: 1 (autonomous vulnerability discovery)
  2026-07-15T13:26:22+00:00 [INFO] candidate topics added: 1 (browser extension)
  2026-07-15T13:26:30+00:00 [INFO] candidate topics added: 2 (agentic ai, oauth security)
  2026-07-15T13:26:33+00:00 [INFO] candidate topics added: 2 (browser extension, supply chain)
  2026-07-15T13:26:33+00:00 [INFO] candidate topics refreshed: 0 candidate

In [ ]:
# Query interface

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML, Markdown
except ImportError:
    widgets = None
    print("ipywidgets is not installed. Use the helper functions directly:")
    print('  ask_summaries_only("...", max_summaries=3)')
    print('  query_annotations("prompt injection", limit=10)')
    print("  analyst_status()")
    print("  list_summary_topics()")
    print("  list_topic_summaries()")

if widgets is not None:
    mode = widgets.ToggleButtons(
        options=[("Question - LLM", "question"), ("Query - no LLM", "query"), ("Topics", "topics"), ("Summary files", "summaries"), ("Status", "status")],
        value="question",
        description="Mode:",
        style={"description_width": "initial"},
    )
    text = widgets.Textarea(
        value="",
        placeholder="Ask a question of the knowledge agent or query the database records.",
        description="Input:",
        layout=widgets.Layout(width="100%", height="110px"),
        style={"description_width": "initial"},
    )
    limit = widgets.IntText(value=10, description="Query limit:", style={"description_width": "initial"})
    max_summaries = widgets.IntSlider(
        value=3,
        min=1,
        max=5,
        step=1,
        description="Max summaries:",
        style={"description_width": "initial"},
    )
    run_button = widgets.Button(description="Run", button_style="primary")
    status = widgets.HTML(
        value="<span style='color:#666'>Idle.</span>",
        layout=widgets.Layout(min_height="24px"),
    )
    output = widgets.Output(layout=widgets.Layout(border="1px solid #ddd", padding="8px"))
    output.add_class("analyst-output-box")
    display(HTML("""
    <style>
      .analyst-output-box .jp-RenderedMarkdown,
      .analyst-output-box .jp-RenderedMarkdown p,
      .analyst-output-box .jp-RenderedMarkdown li,
      .analyst-output-box .jp-RenderedMarkdown h1,
      .analyst-output-box .jp-RenderedMarkdown h2,
      .analyst-output-box .jp-RenderedMarkdown h3,
      .analyst-output-box .jp-RenderedMarkdown h4,
      .analyst-output-box .jp-RenderedMarkdown h5,
      .analyst-output-box .jp-RenderedMarkdown h6 {
        color: var(--jp-content-font-color1, #1f2937) !important;
      }
    </style>
    """))

    def set_status(message: str, color: str = "#666") -> None:
        status.value = f"<span style='color:{color}'>{message}</span>"

    def render_markdown_result(result: str) -> HTML:
        """Render Analyst output as Markdown-styled HTML with explicit text color."""
        safe_result = html.escape(result.replace("```", "'''"))
        try:
            import markdown as markdown_lib

            body = markdown_lib.markdown(safe_result, extensions=["extra", "nl2br"])
        except ImportError:
            body = "<p>" + safe_result.replace("\n", "<br>") + "</p>"
        return HTML(
            """
            <style>
              .analyst-rendered-output,
              .analyst-rendered-output * {
                color: #1f2937 !important;
              }
              .analyst-rendered-output {
                font-family: var(--jp-content-font-family, system-ui, -apple-system, Segoe UI, sans-serif);
                line-height: 1.55;
              }
              .analyst-rendered-output pre,
              .analyst-rendered-output code {
                background: #f3f4f6;
                color: #111827 !important;
              }
            </style>
            <div class="analyst-rendered-output">
            """
            + body
            + "</div>"
        )

    def linkify_plain_text(value: str) -> str:
        """Escape plain text and make URLs clickable."""
        url_re = re.compile(r"https?://[^\s<>()]+")
        parts: list[str] = []
        last = 0
        for match in url_re.finditer(value):
            parts.append(html.escape(value[last:match.start()]))
            url = match.group(0).rstrip(".,;:]")
            trailing = match.group(0)[len(url):]
            safe_url = html.escape(url, quote=True)
            parts.append(
                f'<a href="{safe_url}" target="_blank" rel="noopener noreferrer" '
                f'style="color:#0645ad !important; text-decoration:underline;">'
                f'{html.escape(url)}</a>'
            )
            parts.append(html.escape(trailing))
            last = match.end()
        parts.append(html.escape(value[last:]))
        return "".join(parts)

    def render_text_result(result: str) -> HTML:
        """Render structured database output with clickable URLs and dark text."""
        return HTML(
            '<div style="white-space:pre-wrap; margin:0; '
            'font-family:var(--jp-code-font-family, Consolas, monospace); '
            'line-height:1.45; color:#1f2937 !important;">'
            + linkify_plain_text(result)
            + "</div>"
        )

    def run_analyst(_button):
        run_button.disabled = True
        run_button.description = "Running..."
        set_status("Processing question...", "#2196f3")
        with output:
            output.clear_output()

        try:
            render_as_text = False
            if mode.value == "status":
                set_status("Checking status...", "#2196f3")
                result = analyst_status()
                render_as_text = True
            elif mode.value == "topics":
                set_status("Listing summary topics...", "#2196f3")
                result = list_summary_topics()
                render_as_text = True
            elif mode.value == "summaries":
                set_status("Listing topic summary files...", "#2196f3")
                result = list_topic_summaries()
                render_as_text = True
            elif mode.value == "query":
                set_status("Searching annotations/database...", "#2196f3")
                result = query_annotations(text.value, limit=max(1, int(limit.value)))
                render_as_text = True
            else:
                set_status("NOTE: this agent is a conference analyst and answers questions using the prepared conference topic summaries, not general knowledge. Hit Topics or Summary files for more information. ", "#2196f3")
                result = ask_summaries_only(text.value, max_summaries=max_summaries.value)

            with output:
                output.clear_output()
                display(render_text_result(result) if render_as_text else render_markdown_result(result))
            set_status("Done.", "#2196f3")
        except Exception as exc:
            with output:
                output.clear_output()
                display(Markdown(f"```text\nERROR: {exc}\n```"))
            set_status("Error. See output below.", "#b00020")
        finally:
            run_button.disabled = False
            run_button.description = "Run"

    run_button.on_click(run_analyst)
    display(widgets.VBox([mode, text, limit, max_summaries, run_button, status, output]))

In [5]:
# No-LLM status check.
print(analyst_status())

Records by source:
  blackhat: 110
  bsideslv: 200
  bsidessf: 74
  camlis: 153
  defcon33: 320
  promptorgtfo: 67
  unprompted2026: 64
  total: 988

unclassified: 0
record annotations: 988/988
last import: 2026-06-23T14:10:17+00:00
last classified: 2026-06-23T15:47:28+00:00

Recent import log:
  2026-07-15T13:17:55+00:00 [INFO] candidate topics added: 4 (automation incident response, cloud detection pipeline, analysis bio signal, analysis specialized safety)
  2026-07-15T13:17:58+00:00 [INFO] candidate topics added: 1 (autonomous vulnerability discovery)
  2026-07-15T13:26:09+00:00 [INFO] candidate topics added: 1 (autonomous vulnerability discovery)
  2026-07-15T13:26:22+00:00 [INFO] candidate topics added: 1 (browser extension)
  2026-07-15T13:26:30+00:00 [INFO] candidate topics added: 2 (agentic ai, oauth security)
  2026-07-15T13:26:33+00:00 [INFO] candidate topics added: 2 (browser extension, supply chain)
  2026-07-15T13:26:33+00:00 [INFO] candidate topics refreshed: 0 candidate

In [6]:
# Test / examples of how to make calls::

# No-LLM status check.
print(analyst_status())

# Deterministic, no-LLM annotation query.
print(query_annotations("prompt injection", limit=5))

# List summary-ready topics and existing topic summary files.
print(list_summary_topics())
print(list_topic_summaries())

# Summaries-only LLM answer. Requires provider credentials in .env / environment.
print(ask_summaries_only("What do the summaries say about prompt injection?", max_summaries=3))

Records by source:
  blackhat: 110
  bsideslv: 200
  bsidessf: 74
  camlis: 153
  defcon33: 320
  promptorgtfo: 67
  unprompted2026: 64
  total: 988

unclassified: 0
record annotations: 988/988
last import: 2026-06-23T14:10:17+00:00
last classified: 2026-06-23T15:47:28+00:00

Recent import log:
  2026-07-15T13:17:55+00:00 [INFO] candidate topics added: 4 (automation incident response, cloud detection pipeline, analysis bio signal, analysis specialized safety)
  2026-07-15T13:17:58+00:00 [INFO] candidate topics added: 1 (autonomous vulnerability discovery)
  2026-07-15T13:26:09+00:00 [INFO] candidate topics added: 1 (autonomous vulnerability discovery)
  2026-07-15T13:26:22+00:00 [INFO] candidate topics added: 1 (browser extension)
  2026-07-15T13:26:30+00:00 [INFO] candidate topics added: 2 (agentic ai, oauth security)
  2026-07-15T13:26:33+00:00 [INFO] candidate topics added: 2 (browser extension, supply chain)
  2026-07-15T13:26:33+00:00 [INFO] candidate topics refreshed: 0 candidate